# register-back-fn-after-wrap — ex3: raise contextual KeyError on missing (fwd_fn, argnum)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `register-back-fn-after-wrap`. Running the final beacon cell reports progress against the `Backprop: register back fn` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: register back fn` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`register-back-fn-after-wrap`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "register-back-fn-after-wrap"
DD_SUBTOPIC = "Backprop: register back fn"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## register back fn — quick refresher

`BackwardFuncLookup` is a dict keyed by `(fwd_fn, argnum)`. The dispatcher resolves the back fn at backward-pass time:
```python
back_fn = BACK_FUNCS.get_back_func(recipe.func, argnum)
grad_in = back_fn(grad_out, out, *args, **kwargs)
```
**This drill (ex3) vs prior.** ex1 wired ONE entry and dispatched it. ex2 wired a binary op (two argnums). Both assume the lookup succeeds. ex3 is the FAILURE path: an un-registered op should raise a clear, debuggable `KeyError`, not a silent `None`. The error message must name both the forward fn AND the argnum — that's how you debug "backward through unsupported op" stacks.

### Exercise 3 — raise contextual KeyError on missing (fwd_fn, argnum)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the `(forward_fn, argnum)` lookup pattern with an explicit missing-key path: `get_back_func` raises `KeyError` whose message includes both the fn `__name__` and the argnum.
> Keywords: error-handling, keyerror, missing-registration, debugging-message
> ```

**KCs targeted:** `register-back-fn-after-wrap`, `lookup-missing-raises-keyerror`

Implement `BackwardFuncLookup` with an EXPLICIT missing-key path.

1. `__init__`: create an internal `self._table = {}` (dict keyed by `(fwd_fn, argnum)`).
2. `add_back_func(fwd_fn, argnum, back_fn)`: store the entry.
3. `get_back_func(fwd_fn, argnum)`:
   - If `(fwd_fn, argnum)` is registered, return the back fn.
   - Otherwise raise `KeyError` whose message contains BOTH `fwd_fn.__name__` AND the string `f'argnum={argnum}'`. The test matches both substrings in the exception text.

Then implement `ex3_demo_missing_lookup(BACK_FUNCS, fwd_fn, argnum)` — a small helper that calls `BACK_FUNCS.get_back_func(fwd_fn, argnum)` inside a `try / except KeyError as e` and returns the string form of the exception (so the test can substring-match).

**Why this matters.** A silent `None` from `get_back_func` produces a `'NoneType' is not callable` error one stack frame later — confusing because it doesn't say WHICH op was un-registered. An explicit KeyError with the fn name and argnum saves you 5 minutes of reading tracebacks.

In [ ]:
class BackwardFuncLookup:
    def __init__(self):
        self._table = {}
    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._table[(fwd_fn, argnum)] = back_fn
    def get_back_func(self, fwd_fn, argnum):
        key = (fwd_fn, argnum)
        if key not in self._table:
            name = getattr(fwd_fn, '__name__', repr(fwd_fn))
            raise KeyError(
                f'no back_fn registered for {name} at argnum={argnum}'
            )
        return self._table[key]

def ex3_demo_missing_lookup(BACK_FUNCS, fwd_fn, argnum):
    try:
        BACK_FUNCS.get_back_func(fwd_fn, argnum)
        return ''  # unreachable if registration is correct for the test
    except KeyError as e:
        return str(e)


<details><summary>Solution</summary>

```python
class BackwardFuncLookup:
    def __init__(self):
        self._table = {}
    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._table[(fwd_fn, argnum)] = back_fn
    def get_back_func(self, fwd_fn, argnum):
        key = (fwd_fn, argnum)
        if key not in self._table:
            name = getattr(fwd_fn, '__name__', repr(fwd_fn))
            raise KeyError(
                f'no back_fn registered for {name} at argnum={argnum}'
            )
        return self._table[key]

def ex3_demo_missing_lookup(BACK_FUNCS, fwd_fn, argnum):
    try:
        BACK_FUNCS.get_back_func(fwd_fn, argnum)
        return ''  # unreachable if registration is correct for the test
    except KeyError as e:
        return str(e)
```

**Why an explicit error matters.** ARENA's autograd dispatcher is called for every node during `backward`. If a single op isn't registered, the first symptom you see is whatever the dispatcher does with `None`. Raising a `KeyError` with the function name and argnum gives you a one-line fix path: `add_back_func(that_fn, that_argnum, ...)`.

**Why include `argnum` in the message.** A binary op has TWO entries. "no back_fn for multiply" is ambiguous between argnum=0 and argnum=1 — the actual fix differs (which mathematical derivative to write). Spelling out `argnum=` removes the ambiguity.

**`getattr(fwd_fn, '__name__', repr(fwd_fn))`.** Some forward fns are lambdas or partials with no `__name__`. Falling back to `repr` keeps the error message printable in those cases.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()